In [2]:
import glob
import json
import extrainfo as C

from importlib import reload
from bs4 import BeautifulSoup as Soup

In [3]:
# este es un cfdi normal
# Traslados
# <cfdi:Traslado Base="25000.000000" Impuesto="002" TipoFactor="Tasa" TasaOCuota="0.160000" Importe="4000.000000"/>
# Retenciones 
# <cfdi:Retencion Base="25000.000000" Impuesto="002" TipoFactor="Tasa" TasaOCuota="0.106667" Importe="2666.675000"/>
# <cfdi:Retencion Base="25000.000000" Impuesto="001" TipoFactor="Tasa" TasaOCuota="0.100000" Importe="2500.000000"/>
archivo_nrml = 'poncho/00f53dfe-fa00-4001-8786-f0596c4d6abe.xml'
# este tiene pago_20
# pago20:DoctoRelacionado IdDocumento="B442A737-D76C-490B-90FB-A8C247A9A0C9" 
# <cfdi:Retencion Impuesto="002" Importe="2666.68"/>
# <cfdi:Retencion Impuesto="001" Importe="2500.00"/>
# <cfdi:Traslado Base="25000.00" Impuesto="002" TipoFactor="Tasa" TasaOCuota="0.160000" Importe="4000.00"/>
archivo_p20 = 'poncho/1f76c534-8559-427d-8bf3-a52fe0c3352c.xml'
# varios conceptos y traslados
archivo_v = 'poncho/5d811066-8720-4d85-ab63-fb09042082c1.xml'
# archivo_v = 'poncho/8f9f345e-67dc-4053-9221-a04565b38ff4.xml"'
archivo_v = "poncho/3911a9cf-09f3-495e-8f2f-d4d6d5d563b4.xml"

Normal

In [56]:
soup = Soup(open(archivo_nrml, 'r').read(), 'lxml')
# soup
impuestos = list(soup.find('cfdi:impuestos').children)
print(impuestos)


traslados = [
    traslado
    for cont in impuestos
    if getattr(cont, 'name', None) == 'cfdi:traslados'
    for traslado in cont.find_all('cfdi:traslado', recursive=False)
]

retenciones = [
    retencion
    for cont in impuestos
    if getattr(cont, 'name', None) == 'cfdi:retenciones'
    for retencion in cont.find_all('cfdi:retencion', recursive=False)
]

retenciones, traslados



['\n', <cfdi:traslados>
<cfdi:traslado base="25000.000000" importe="4000.000000" impuesto="002" tasaocuota="0.160000" tipofactor="Tasa"></cfdi:traslado>
</cfdi:traslados>, '\n', <cfdi:retenciones>
<cfdi:retencion base="25000.000000" importe="2666.675000" impuesto="002" tasaocuota="0.106667" tipofactor="Tasa"></cfdi:retencion>
<cfdi:retencion base="25000.000000" importe="2500.000000" impuesto="001" tasaocuota="0.100000" tipofactor="Tasa"></cfdi:retencion>
</cfdi:retenciones>, '\n']


([<cfdi:retencion base="25000.000000" importe="2666.675000" impuesto="002" tasaocuota="0.106667" tipofactor="Tasa"></cfdi:retencion>,
  <cfdi:retencion base="25000.000000" importe="2500.000000" impuesto="001" tasaocuota="0.100000" tipofactor="Tasa"></cfdi:retencion>],
 [<cfdi:traslado base="25000.000000" importe="4000.000000" impuesto="002" tasaocuota="0.160000" tipofactor="Tasa"></cfdi:traslado>])

In [68]:
reload(C)

<module 'extrainfo' from '/mnt/c/Users/usuario/Documents/dev/lectorcfdi/extrainfo.py'>

In [69]:
cfdi = C.CFDI(archivo_nrml)
cfdi.dic_cfdi

CFDI normal


{'Folio_fiscal': '00F53DFE-FA00-4001-8786-F0596C4D6ABE',
 'Fecha_CFDI': '2026-03-19T17:37:58',
 'Tipo': 'Ingreso',
 'RFC_Emisor': 'COPL750415DR0',
 'Emisor': 'LUIS EMILIO CORTES PIÑON',
 'RFC_Receptor': 'CLM110216R54',
 'Receptor': 'CONCEPTO LIBRE MEXICANO',
 'Total': 23833.32,
 'IVA': 0.0,
 'Ret IVA': 2500.0,
 'Forma_pago': '03',
 'Moneda': 'MXN',
 'Complemento': '',
 'ISR': 4000.0,
 'Ret ISR': 2666.68}

Con varios conceptos o característica especil

In [78]:
soup = Soup(open(archivo_v, 'r').read(), 'lxml')
print(archivo_v)
impuestos = list(soup.find('cfdi:impuestos').children)
print(impuestos)
retenciones = [x for x in impuestos if getattr(x, 'name', None) in ['cfdi:retencion']]
traslados   = [x for x in impuestos if getattr(x, 'name', None) in ['cfdi:traslado']]

poncho/3911a9cf-09f3-495e-8f2f-d4d6d5d563b4.xml
['\n', <cfdi:traslados>
<cfdi:traslado base="128798.23" impuesto="002" tipofactor="Exento"></cfdi:traslado>
</cfdi:traslados>, '\n']


In [75]:
traslado = impuestos[1].find('cfdi:traslado')
traslado['tipofactor']

'Exento'

In [70]:

cfdi = C.CFDI(archivo_v)
cfdi.dic_cfdi

CFDI normal


{'Folio_fiscal': '3911a9cf-09f3-495e-8f2f-d4d6d5d563b4',
 'Fecha_CFDI': '2026-03-18T21:51:53',
 'Tipo': 'Ingreso',
 'RFC_Emisor': 'BSM970519DU8',
 'Emisor': 'BANCO SANTANDER MEXICO S.A., INSTITUCION DE BANCA MULTIPLE, GRUPO FINANCIERO SANTANDER MEXICO',
 'RFC_Receptor': 'CLM110216R54',
 'Receptor': 'CONCEPTO LIBRE MEXICANO',
 'Total': 128798.23,
 'IVA': 0.0,
 'Ret IVA': 0.0,
 'Forma_pago': '03',
 'Moneda': 'MXN',
 'Complemento': '',
 'ISR': 0.0,
 'Ret ISR': 0.0}

Con pago 20

In [71]:
cfdi = C.CFDI(archivo_p20)
cfdi.dic_cfdi

Pago20
Lonitud uno


{'Folio_fiscal': '1F76C534-8559-427D-8BF3-A52FE0C3352C',
 'Fecha_CFDI': '2026-03-03T22:43:45',
 'Tipo': 'Pagado',
 'RFC_Emisor': 'VLE060918B86',
 'Emisor': 'VOLKSWAGEN LEASING',
 'RFC_Receptor': 'CLM110216R54',
 'Receptor': 'CONCEPTO LIBRE MEXICANO',
 'Total': 0.0,
 'IVA': 0.0,
 'Ret IVA': 0.0,
 'Forma_pago': 'Pagado',
 'Moneda': 'XXX',
 'Complemento': 'B442A737-D76C-490B-90FB-A8C247A9A0C9',
 'ISR': 1897.58,
 'Ret ISR': 0.0}

In [96]:
d = {'a' : 1, 'b' : 3,'c' : 5}


KeyError: 0

In [93]:
archivo_p20_2 = 'poncho/97d9ebb6-b689-4f8b-8f36-480fd8d68835.xml'
cfdi = C.CFDI(archivo_p20_2)
cfdi.dic_cfdi

Pago20
Lonitud uno
Traslados [<pago20:trasladop basep="14625.71" importep="2340.11" impuestop="002" tasaocuotap="0.160000" tipofactorp="Tasa"></pago20:trasladop>]


KeyError: 'impuesto'